# Utility Placement Optimisation

**Learning outcome:** Apply utility placement optimisation through the public `PinchProblem` or `PinchWorkspace` workflow.

**Level:** Advanced  
**Execution profile:** `base`  
**Expected runtime:** under 2 minutes  
**Optional extras:** plot

The lifecycle is explicit: prepare the study, run the named method, then inspect cached results. Observation cells do not launch analysis.

## Study question and data

**Study question:** Where should two isothermal and two sensible hot and cold utility levels be placed at Process and Site hierarchy levels to minimize thermodynamic cost?

The sample data is packaged with OpenPinch, so the notebook runs without path setup. Read the named inputs and assumptions before substituting plant data.

## Step 1: Prepare the placement study

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [1]:
import pandas as pd

from OpenPinch import PinchWorkspace

workspace = PinchWorkspace(
    source="chocolate_factory.json", project_name="Site"
)
problem = workspace.use_case("baseline")
baseline_input = problem.to_problem_json()

def retarget_comparison(evidence, target):
    period = evidence.best.period_results[0]
    rows = []
    for side, levels, utilities in (
        ("hot", period.hot_levels, target.hot_utilities),
        ("cold", period.cold_levels, target.cold_utilities),
    ):
        retargeted = {utility.name: float(utility.heat_flow.value) for utility in utilities}
        for level in levels:
            name = level.template_key.name
            rows.append({
                "side": side,
                "utility": name,
                "kind": level.kind.value,
                "supply_degC": level.supply_temperature.value,
                "target_degC": level.target_temperature.value,
                "optimizer_duty_kW": level.allocated_duty.value,
                "retargeted_duty_kW": retargeted.get(name, 0.0),
                "difference_kW": retargeted.get(name, 0.0) - level.allocated_duty.value,
                "fallback": level.is_fallback,
            })
    return pd.DataFrame(rows)

search_options = {
    "iteration_limit": 1,
    "evaluation_limit": 100,
    "candidate_limit": 2,
    "run_count": 1,
    "minimum_sensible_span": {"value": 10.0, "unit": "delta_degC"},
}
process_maximum_duties = {
    f"hot_{suffix}": 20.0
    for suffix in ("iso_1", "iso_2", "sensible_1", "sensible_2")
}

## Step 2: Optimize a Process Zone and build its standard GCC

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [2]:
process_case = problem.target.utility_placement(
    isothermal=2,
    sensible=2,
    zone="Almond",
    period_ids=("0",),
    maximum_duties=process_maximum_duties,
    options=search_options,
)
process_evidence = process_case.utility_placement_result
process_objective = process_evidence.best.aggregate_objective
process_fallback_penalty = process_evidence.best.fallback_penalty
process_utilities = process_case.to_problem_json()["utilities"]
process_case = workspace.add(
    process_case,
    name="optimized_process_utilities",
    activate=False,
)
process_target = process_case.target.direct_heat_integration(
    zone="Almond", period_id="0"
)
process_retarget_comparison = retarget_comparison(
    process_evidence, process_target
)
process_summary = process_case.summary_frame()
process_gcc = process_case.plot.grand_composite_curve(
    zone_name="Almond"
)

## Step 3: Optimize the Site and build its standard Total Site Profile

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [3]:
site_case = problem.target.utility_placement(
    isothermal=2,
    sensible=2,
    period_ids=("0",),
    options=search_options,
)
site_evidence = site_case.utility_placement_result
site_objective = site_evidence.best.aggregate_objective
site_utilities = site_case.to_problem_json()["utilities"]
site_case = workspace.add(
    site_case,
    name="optimized_site_utilities",
    activate=False,
)
baseline_unchanged = workspace.use_case("baseline").to_problem_json() == baseline_input
site_target = site_case.target.total_site_heat_integration(period_id="0")
site_retarget_comparison = retarget_comparison(site_evidence, site_target)
site_summary = site_case.summary_frame()
site_tsp = site_case.plot.total_site_profiles()

## Review the result

Review both optimized cases exactly like normal cases: compare the Process utilities on the standard GCC with the Site utilities on the standard Total Site Profile. The Process Utility GCC must not cross the Process GCC. The comparison tables place optimizer-evidence and ordinary-retarget duties side by side; a zero difference shows exact replay.

In [4]:
from IPython.display import display

display(process_objective)
display(process_fallback_penalty)
display(process_retarget_comparison)
display(process_summary)
display(process_gcc)
display(site_objective)
display(site_retarget_comparison)
display({"baseline_unchanged": baseline_unchanged})
display(site_summary)
display(site_tsp)

QuantityValue(value=0.05231848324277155, unit='kW/K')

QuantityValue(value=1.1809269259983282, unit='dimensionless')

,side,utility,kind,supply_degC,target_degC,optimizer_duty_kW,retargeted_duty_kW,difference_kW,fallback
0,hot,hot_iso_1,isothermal,162.878925,162.868925,20.0000,20.0000,0.000000e+00,False
1,hot,hot_iso_2,isothermal,138.166430,138.156430,20.0000,20.0000,0.000000e+00,False
2,hot,hot_sensible_1,sensible,115.692620,97.811007,20.0000,20.0000,0.000000e+00,False
3,hot,hot_sensible_2,sensible,69.920742,52.237055,20.0000,20.0000,0.000000e+00,False
4,hot,HU,isothermal,178.000000,177.990000,59.2164,59.2164,0.000000e+00,True
5,cold,cold_iso_1,isothermal,162.868925,162.878925,0.0000,0.0000,0.000000e+00,False
6,cold,cold_iso_2,isothermal,138.156430,138.166430,0.0000,0.0000,0.000000e+00,False
7,cold,cold_sensible_1,sensible,97.811007,115.692620,0.0000,0.0000,0.000000e+00,False
8,cold,cold_sensible_2,sensible,52.237055,69.920742,0.0000,0.0000,0.000000e+00,False
9,cold,CU,isothermal,17.000000,17.010000,12.1363,12.1363,1.421085e-14,True


,Scope,Zone Type,Integration Type,Target Method,Period ID,Hot Utility Target,Cold Utility Target,Heat Recovery,Hot Pinch,Cold Pinch,Hot Utilities,Cold Utilities
0,Site/Almond,Process Zone,Process,Heat Exchange,0,139.22 kW,12.14 kW,51.35 kW,33.00 degC,33.00 degC,"HU: 59.22 kW, hot_iso_1: 20.00 kW, hot_iso_2: ...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."


QuantityValue(value=0.5483550713205982, unit='kW/K')

,side,utility,kind,supply_degC,target_degC,optimizer_duty_kW,retargeted_duty_kW,difference_kW,fallback
0,hot,hot_iso_1,isothermal,183.010000,183.000000,35.526258,35.526258,0.0,False
1,hot,hot_iso_2,isothermal,156.506667,156.496667,451.935497,451.935497,0.0,False
2,hot,hot_sensible_1,sensible,130.003333,86.401333,89.541137,89.541137,0.0,False
3,hot,hot_sensible_2,sensible,103.500000,59.898000,754.434285,754.434285,0.0,False
4,cold,cold_iso_1,isothermal,183.000000,183.010000,0.000000,0.000000,0.0,False
5,cold,cold_iso_2,isothermal,156.496667,156.506667,0.000000,0.000000,0.0,False
6,cold,cold_sensible_1,sensible,86.401333,130.003333,0.000000,0.000000,0.0,False
7,cold,cold_sensible_2,sensible,59.898000,103.500000,0.000000,0.000000,0.0,False
8,cold,CU,isothermal,17.000000,17.010000,3650.593120,3650.593120,0.0,True


{'baseline_unchanged': True}

,Scope,Zone Type,Integration Type,Target Method,Period ID,Hot Utility Target,Cold Utility Target,Heat Recovery,Hot Pinch,Cold Pinch,Hot Utilities,Cold Utilities
0,Site,Site,Process,Heat Exchange,0,1082.54 kW,3401.69 kW,440.47 kW,33.00 degC,33.00 degC,"hot_iso_1: 35.53 kW, hot_iso_2: 497.40 kW, hot...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."
1,Site,Site,Utility,Heat Exchange,0,1331.44 kW,3650.59 kW,191.57 kW,59.90 degC,17.01 degC,"hot_iso_1: 35.53 kW, hot_iso_2: 451.94 kW, hot...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."
2,Site/Almond,Process Zone,Process,Heat Exchange,0,139.22 kW,12.14 kW,51.35 kW,33.00 degC,33.00 degC,"hot_iso_1: 17.14 kW, hot_iso_2: 76.81 kW, hot_...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."
3,Site/Blending,Process Zone,Process,Heat Exchange,0,1.16 kW,1.38 kW,0.79 kW,41.00 degC,41.00 degC,"hot_iso_1: 0.00 kW, hot_iso_2: 0.00 kW, hot_se...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."
4,Site/CIP,Process Zone,Process,Heat Exchange,0,13.88 kW,0.00 kW,0.00 kW,12.00 degC,12.00 degC,"hot_iso_1: 0.00 kW, hot_iso_2: 0.00 kW, hot_se...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."
5,Site/Cocoa,Process Zone,Process,Heat Exchange,0,139.92 kW,19.86 kW,88.46 kW,77.00 degC,33.00 degC,"hot_iso_1: 5.40 kW, hot_iso_2: 47.85 kW, hot_s...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."
6,Site/Compressor,Process Zone,Process,Heat Exchange,0,0.00 kW,48.66 kW,0.00 kW,66.00 degC,66.00 degC,"hot_iso_1: 0.00 kW, hot_iso_2: 0.00 kW, hot_se...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."
7,Site/Conching,Process Zone,Process,Heat Exchange,0,2.88 kW,78.28 kW,27.14 kW,76.00 degC,76.00 degC,"hot_iso_1: 0.00 kW, hot_iso_2: 0.00 kW, hot_se...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."
8,Site/Debacterization,Process Zone,Process,Heat Exchange,0,22.07 kW,0.00 kW,0.00 kW,38.00 degC,38.00 degC,"hot_iso_1: 0.00 kW, hot_iso_2: 0.00 kW, hot_se...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."
9,Site/Hazelnut,Process Zone,Process,Heat Exchange,0,222.16 kW,14.34 kW,0.00 kW,33.00 degC,33.00 degC,"hot_iso_1: 12.98 kW, hot_iso_2: 86.97 kW, hot_...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."


## Interpret the result

Compare the Process result against its direct GCC and the Site result against its Total Site Profile. Candidate duties come from those exact ordinary target workflows; they are not independent optimizer decisions, and a requested level may be unused. Inspect physical entropy generation from the balanced composite curves: use CP * ln(T_out / T_in) in kelvin for sensible intervals and the signed Q / T limit for isothermal intervals. In the capped Process example, confirm that exact targeting limits sensible duty at every GCC breakpoint, so the Utility GCC cannot cross the Process GCC, before assigning only the remaining shortfall to HU.

## Adapt this template

Replace the sample with validated plant data, apply defensible temperature bounds, and increase the optimizer limits before making an engineering decision.

Keep the workflow explicit: prepare input, call one named engineering method, inspect cached results, then export.